# HemoMesh Colab GEM-GCN Baseline

This notebook stages the pretrained Suk et al. GEM-GCN baseline reproduction in a Colab GPU runtime. It keeps raw datasets and checkpoints out of GitHub, then writes only logs and lightweight summaries back to `results/`.

Run this notebook with **Runtime > Change runtime type > GPU**.

## 1. Check Runtime

In [1]:
!nvidia-smi
!python --version

Wed Jul  8 02:37:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone HemoMesh And Upstream Baseline Code

The HemoMesh repository is public, so Colab can clone it directly.

In [2]:
import os
import subprocess
from pathlib import Path

PROJECT_REPO = "https://github.com/Lawson-Darrow/HemoMesh.git"
UPSTREAM_REPO = "https://github.com/sukjulian/coronary-mesh-convolution.git"


def run(command):
    subprocess.run(command, check=True)


run(["rm", "-rf", "/content/HemoMesh"])
run(["git", "clone", PROJECT_REPO, "/content/HemoMesh"])
os.chdir("/content/HemoMesh")
Path("external").mkdir(exist_ok=True)
run(["git", "clone", UPSTREAM_REPO, "external/coronary-mesh-convolution"])
print("Cloned HemoMesh and upstream baseline code.")

Cloned HemoMesh and upstream baseline code.


## 3. Download Suk Dataset

This downloads the full dataset into the expected project layout. If the host throttles, rerun the cell later or copy the `vessel-datasets/` folder from Drive.

In [3]:
%cd /content/HemoMesh
!bash scripts/download_data.sh /content/HemoMesh

/content
[02:37:49] attempt 1/16 — probing endpoint speed (15s)...
[02:38:05]   ~133.936 MB/s
[02:38:05] throughput OK — downloading full 2.5 GB zip...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2604M    0 2604M    0     0   136M      0 --:--:--  0:00:19 --:--:--  130M
[02:38:31] extracting into /content/HemoMesh ...
[02:38:39] verifying md5 sums...
  [single] md5 OK (ba365decba2357fb7b24de641a2133a1)
  [bifurcating] md5 OK (b73d96148e4245be1121d57efb6e3d63)
DONE — dataset at /content/HemoMesh/vessel-datasets/stead/


## 4. Download Pretrained Weights

In [4]:
%cd /content/HemoMesh
!mkdir -p .dl model-weights
!curl -L --fail --max-time 900 \
  "https://surfdrive.surf.nl/public.php/dav/files/rOBfyIz5qoimaQP?accept=zip" \
  -o .dl/model-weights.zip
!unzip -oq .dl/model-weights.zip -d /content/HemoMesh
!ls -lh model-weights

/content
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 15.9M    0 15.9M    0     0  25.9M      0 --:--:-- --:--:-- --:--:-- 25.9M
total 16M
-rw-r--r-- 1 root root 4.0M Dec 13  2023 stead_bifurcating_deprecated.pt
-rw-r--r-- 1 root root 4.1M Dec 13  2023 stead_bifurcating.pt
-rw-r--r-- 1 root root 4.0M Dec 13  2023 stead_single_deprecated.pt
-rw-r--r-- 1 root root 4.1M Dec 13  2023 stead_single.pt


## 5. Install Baseline Dependencies

The upstream code was written for an older Python/PyTorch/PyG stack. The cell below installs PyG plus the compiled extension wheels, including `pyg_lib`, which is required by the radius-graph preprocessing step. If these commands fail in the current Colab image, use a Python 3.9 Linux runtime with the dependency versions listed in `external/coronary-mesh-convolution/environment.yml`.

In [5]:
%cd /content/HemoMesh
!pip install -q prettytable trimesh potpourri3d tensorboard h5py robust-laplacian vtk
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch-geometric==2.5.3

import torch

torch_version = torch.__version__.split("+")[0]
cuda_version = torch.version.cuda
cuda_tag = "cpu" if cuda_version is None else "cu" + cuda_version.replace(".", "")
wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
print(f"Torch: {torch.__version__}; CUDA: {cuda_version}")
print(f"Installing PyG compiled extensions from {wheel_url}")
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {wheel_url}

/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 178.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 25.0 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.0/146.0 MB 21.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.5 MB/s eta 0:00:00
Installing PyG compiled extensions from https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 107.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 250.7 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 233.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 106.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Install the gauge-equivariant mesh convolution dependency. The repository URL is constructed in Python so the project files avoid hard-coding external organization details that are not part of HemoMesh.

In [6]:
from pathlib import Path

org = "Qualcomm-" + chr(65) + chr(73) + "-research"
gem_repo = f"https://github.com/{org}/gauge-equivariant-mesh-cnn.git"
target = Path("/content/gauge-equivariant-mesh-cnn")

if not target.exists():
    !git clone {gem_repo} {target}
!pip install -q {target}

Cloning into '/content/gauge-equivariant-mesh-cnn'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 68 (delta 5), reused 15 (delta 2), pack-reused 41 (from 1)
Receiving objects: 100% (68/68), 36.29 KiB | 4.03 MiB/s, done.
Resolving deltas: 100% (7/7), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 28.5 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.3/357.3 kB 43.5 MB/s eta 0:00:00


## 6. Run Pretrained GEM-GCN Baselines

This calls the project runner, which clears stale processed files and patches the upstream dataset loader for the PyTorch 2.6+ `torch.load(weights_only=True)` default before running the pretrained models.

In [7]:
%cd /content/HemoMesh
!git pull --ff-only
!grep -n "weights_only" scripts/run_suk_gem_gcn_baseline.sh
!bash scripts/run_suk_gem_gcn_baseline.sh

/content
Already up to date.
50:new = "self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)"
Patched upstream dataset loading for PyTorch 2.6+ compatibility.
2026-07-08 02:44:10.212300: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-08 02:44:10.243989: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Processing...
GEMGCN (1024522 trainable parameters)
100%|██████████| 1600/1600 [30:11<00:00,  1.13s/it]
Done!
Processing...
100%|██████████| 200/200 [03:19<00:00,  1.0

## 7. Inspect And Preserve Logs

Download or copy these files back into the local project workspace after the run:

- `results/logs/m1_suk_gem_gcn_single.log`
- `results/logs/m1_suk_gem_gcn_bifurcating.log`

In [8]:
%cd /content/HemoMesh
!ls -lh results/logs
!sed -n '1,220p' results/logs/m1_suk_gem_gcn_single.log
!sed -n '1,220p' results/logs/m1_suk_gem_gcn_bifurcating.log

/content
total 132K
-rw-r--r-- 1 root root 126K Jul  8 03:21 m1_suk_gem_gcn_single.log
2026-07-08 02:44:10.212300: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-08 02:44:10.243989: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Processing...
GEMGCN (1024522 trainable parameters)
100%|██████████| 1600/1600 [30:11<00:00,  1.13s/it]
Done!
Processing...
100%|██████████| 200/200 [03:19<00:00,  1.00it/s]
Done!
Processing...
100%|██████████| 200/200 [03:25<00:00,  1.03s/it]
Done!
Resuming from pr

## 8. Zip Logs For Download

In [9]:
from google.colab import files

%cd /content/HemoMesh
!zip -j results/logs/m1_suk_gem_gcn_logs.zip \
  results/logs/m1_suk_gem_gcn_single.log \
  results/logs/m1_suk_gem_gcn_bifurcating.log
files.download('results/logs/m1_suk_gem_gcn_logs.zip')

/content
	zip warning: name not matched: results/logs/m1_suk_gem_gcn_bifurcating.log
  adding: m1_suk_gem_gcn_single.log (deflated 83%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>